# Zusatzspalten-Bereinigung + NOVA-Ableitung — erklärt

Dieses Notebook **bereinigt** eine Spaltengruppe und **leitet** danach fehlende NOVA-Gruppen ab.
Alles läuft in **einem** `df`: der NOVA-Teil lädt **nicht** neu, sondern arbeitet auf den gereinigten
Daten weiter. Am Ende enthält `df` die gereinigten Spalten **und** `nova_group_derived` +
`nova_group_source`. Zelle für Zelle von oben nach unten ausführen.

**Setup:** lädt pandas/numpy/matplotlib, zeigt alle Spalten an.

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
pd.set_option("display.max_columns", None)

## Bereinigung (Herzstück)

Lädt die volle CSV (nur nötige Spalten via `usecols`) und definiert die Helfer:
`clean_series` (Junk→fehlend, Präfixe bleiben), `merge_report`/`clean_report` (Log-Zeilen),
`normalize_tags` (nur packaging: Tags→lesbar, Präfixe weg). Dann drei Teile:

- **Teil A** – `product_name`, `packaging`, `additives`, `ingredients` (+`ingredients_text`), `manufacturing_places`.
- **Teil B** – `ingredients_analysis_tags`, `nova_group` (→ 1–4, `Int64`), `nutrient_levels_tags`.
- **Teil C** – `cities_tags`, `owner`, `traces`/`traces_tags`/`traces_en` (Präfixe bleiben), `owner` minimal.

Zum Schluss `rename(...)` der gemergten Spalten. **Wichtig:** `additives_tags` und `categories_tags`
bleiben im `df` — der NOVA-Teil braucht diese Tag-Formen (`additives` ist die lesbare Form, taugt nicht
fürs E-Nummern-Matching; `ingredients` IST hingegen die umbenannte Tag-Spalte und wird direkt genutzt).

In [2]:
# Volle CSV laden — nur die fuer diese Spaltengruppe noetigen Spalten (inkl. Quell-Varianten fuer die Merges).
# Bewusst die VOLLE Datei (nicht der >6%-Datensatz): cities_tags / owner / traces* liegen unter 6 %
# und fehlen in openfoodfacts_ueber6prozent.csv.zip.
CSV_PATH = "../en.openfoodfacts.org.products 2.csv"
COLS = [
    "product_name", "abbreviated_product_name", "generic_name",
    "packaging_en", "packaging", "packaging_tags", "packaging_text",
    "additives_en", "additives_tags", "additives",
    "ingredients_tags", "ingredients_text",
    "manufacturing_places_tags", "manufacturing_places",
    "ingredients_analysis_tags", "nova_group", "nutrient_levels_tags",
    "cities_tags", "owner", "traces", "traces_tags", "traces_en", "categories_tags",
]

df = pd.read_csv(CSV_PATH, sep="\t", usecols=COLS, low_memory=False, on_bad_lines="skip")
df = df.copy()

# ── Helper ────────────────────────────────────────────────────────────────────
# Junk-Platzhalter -> fehlend. Behaelt bewusst Praefixe wie "en:"/"fr:".
INVALID = {"?", ".", ",", "n-a", "na", "none", "null", "0", "en:null", "en:none"}

def clean_series(s):
    s = s.astype("string").str.strip()
    s = s.replace("", pd.NA)
    s = s.where(~s.str.lower().isin(INVALID), pd.NA)
    s = s.where(~s.str.fullmatch(r"[?,.\-/ ]+", na=False), pd.NA)
    return s

_W = 68

def section_header(title):
    print(f"\n{'═' * _W}")
    print(f"  {title}")
    print(f"{'─' * _W}")

def merge_report(col_name, before, after):
    pct   = after / len(df) * 100
    added = after - before
    bar   = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<26}  [{bar}]  {pct:5.1f}%   befuellt {after:,}  (+{added:,})")

def clean_report(col_name, before, after):
    pct     = after / len(df) * 100
    removed = before - after
    bar     = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<26}  [{bar}]  {pct:5.1f}%   befuellt {after:,}  (entfernt {removed:,})")

# Taxonomie-Tags -> lesbare Form, Praefixe ENTFERNT (nur fuer packaging, wie im Hauptnotebook)
def normalize_tags(s):
    return (
        clean_series(s.astype(str).where(s.notna(), np.nan))
        .str.replace(r"\b[a-z]{2}:", "", regex=True)
        .str.replace(r"[-_]", " ", regex=True)
        .str.strip()
        .str.title()
    )


# ── Manufacturing-Places-Helfer: Ländernamen -> Englisch, Regionen/Städte als Eigennamen behalten ──
# Multilinguales Länder-Wörterbuch (Schlüssel deakzentuiert+lowercase) -> englischer Name.
# Regionen/Städte werden NICHT aufs Land kollabiert, zählen aber für die Erkennungs-Quote.
def _deaccent(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def _mfg_key(tok):
    k = re.sub(r"^[a-z]{2}:", "", _deaccent(tok).lower())
    k = re.sub(r"[-_]", " ", k)
    return re.sub(r"\s+", " ", k).strip()

def _mfg_C(en, *keys):
    return {_mfg_key(k): en for k in keys}

COUNTRY_EN = {}
for _d in [
    _mfg_C("France", "france", "francia", "frankreich", "francie", "frankrijk", "francja"),
    _mfg_C("Germany", "germany", "deutschland", "allemagne", "alemania", "germania", "niemcy", "nemecko", "duitsland", "alemanha"),
    _mfg_C("Switzerland", "switzerland", "suisse", "schweiz", "svizzera", "suiza", "suica", "zwitserland"),
    _mfg_C("Italy", "italy", "italie", "italia", "italien", "wlochy"),
    _mfg_C("Spain", "spain", "espagne", "espana", "spanien", "spagna", "espanha", "hiszpania", "spanje"),
    _mfg_C("Belgium", "belgium", "belgique", "belgie", "belgien", "belgio", "belgia"),
    _mfg_C("Mexico", "mexico", "mexique", "mexiko"),
    _mfg_C("United States", "united states", "usa", "u s a", "etats unis", "estados unidos", "vereinigte staaten", "united states of america", "stati uniti"),
    _mfg_C("United Kingdom", "united kingdom", "uk", "u k", "royaume uni", "reino unido", "grossbritannien", "great britain", "angleterre", "england", "regno unito", "verenigd koninkrijk"),
    _mfg_C("Czech Republic", "czech republic", "cesko", "ceska republika", "tschechien", "republique tcheque", "czechia", "repubblica ceca", "czechy"),
    _mfg_C("Austria", "austria", "osterreich", "autriche", "rakousko", "oostenrijk"),
    _mfg_C("Netherlands", "netherlands", "pays bas", "nederland", "niederlande", "holland", "paesi bassi", "paises bajos"),
    _mfg_C("Poland", "poland", "pologne", "polska", "polen", "polsko"),
    _mfg_C("Tunisia", "tunisia", "tunisie", "tunesien", "tunez"),
    _mfg_C("Norway", "norway", "norvege", "norwegen", "norge", "noruega"),
    _mfg_C("Australia", "australia", "australie", "australien"),
    _mfg_C("Argentina", "argentina", "argentine", "argentinien"),
    _mfg_C("Thailand", "thailand", "thailande", "tailandia"),
    _mfg_C("China", "china", "chine", "cina", "chiny"),
    _mfg_C("Romania", "romania", "roumanie", "rumanien", "rumania"),
    _mfg_C("Portugal", "portugal", "portogallo"),
    _mfg_C("Canada", "canada", "kanada"),
    _mfg_C("Sweden", "sweden", "suede", "schweden", "sverige", "suecia", "szwecja"),
    _mfg_C("Denmark", "denmark", "danemark", "danmark", "dinamarca"),
    _mfg_C("Finland", "finland", "finlande", "suomi", "finlandia", "finnland"),
    _mfg_C("Greece", "greece", "grece", "griechenland", "grecia", "grecja"),
    _mfg_C("Turkey", "turkey", "turquie", "turkei", "turkiye", "turchia", "turquia"),
    _mfg_C("Japan", "japan", "japon", "giappone", "japonia", "japonsko"),
    _mfg_C("India", "india", "inde", "indien"),
    _mfg_C("Brazil", "brazil", "bresil", "brasilien", "brasil", "brasile", "brazylia"),
    _mfg_C("Russia", "russia", "russie", "russland", "rusia", "rosja", "rusko"),
    _mfg_C("Ireland", "ireland", "irlande", "irland", "irlanda"),
    _mfg_C("Hungary", "hungary", "hongrie", "ungarn", "madarsko", "magyarorszag", "hungria"),
    _mfg_C("Slovakia", "slovakia", "slovaquie", "slowakei", "slovensko", "eslovaquia"),
    _mfg_C("Slovenia", "slovenia", "slovenie", "slowenien", "slovenija", "eslovenia"),
    _mfg_C("Croatia", "croatia", "croatie", "kroatien", "hrvatska"),
    _mfg_C("Morocco", "morocco", "maroc", "marokko", "marruecos"),
    _mfg_C("Luxembourg", "luxembourg", "luxemburg", "lussemburgo"),
    _mfg_C("Ukraine", "ukraine", "ukrajina", "ucrania"),
    _mfg_C("New Zealand", "new zealand", "nouvelle zelande", "neuseeland"),
    _mfg_C("South Korea", "south korea", "coree du sud", "korea", "sudkorea", "corea del sur"),
    _mfg_C("Vietnam", "vietnam", "viet nam"),
    _mfg_C("Indonesia", "indonesia", "indonesie", "indonesien"),
    _mfg_C("Serbia", "serbia", "serbie", "srbija"),
    _mfg_C("Bulgaria", "bulgaria", "bulgarie", "bulgarien"),
    _mfg_C("Lithuania", "lithuania", "lituanie", "lietuva", "litauen"),
    _mfg_C("Latvia", "latvia", "lettonie", "letland", "lettland"),
    _mfg_C("Estonia", "estonia", "estonie", "eesti", "estland"),
    _mfg_C("Egypt", "egypt", "egypte", "agypten", "egipto"),
    _mfg_C("Israel", "israel"),
    _mfg_C("Colombia", "colombia", "colombie", "kolumbien"),
    _mfg_C("Chile", "chile", "chili"),
    _mfg_C("Peru", "peru", "perou"),
    _mfg_C("Bolivia", "bolivia", "bolivie", "bolivien"),
    _mfg_C("Ecuador", "ecuador", "equateur"),
    _mfg_C("Uruguay", "uruguay"),
    _mfg_C("Paraguay", "paraguay"),
    _mfg_C("Venezuela", "venezuela"),
    _mfg_C("South Africa", "south africa", "afrique du sud", "sudafrika", "sudafrica"),
    _mfg_C("Iceland", "iceland", "islande", "island", "islandia"),
    _mfg_C("Algeria", "algeria", "algerie", "argelia"),
    _mfg_C("Philippines", "philippines", "filipinas", "filippine"),
    _mfg_C("Singapore", "singapore", "singapour", "singapur"),
    _mfg_C("Malaysia", "malaysia", "malaisie", "malasia"),
    _mfg_C("Sri Lanka", "sri lanka"),
    _mfg_C("Lebanon", "lebanon", "liban", "libano"),
    _mfg_C("Saudi Arabia", "saudi arabia", "arabie saoudite", "arabia saudita"),
    _mfg_C("United Arab Emirates", "united arab emirates", "emirats arabes unis", "uae"),
    _mfg_C("Pakistan", "pakistan"),
    _mfg_C("Iran", "iran"),
    _mfg_C("Cyprus", "cyprus", "chypre", "zypern", "cipro"),
    _mfg_C("Malta", "malta", "malte"),
    _mfg_C("Costa Rica", "costa rica"),
    _mfg_C("Guatemala", "guatemala"),
    _mfg_C("Senegal", "senegal"),
    _mfg_C("Ivory Coast", "ivory coast", "cote d ivoire", "costa de marfil"),
    _mfg_C("Cameroon", "cameroon", "cameroun"),
    _mfg_C("Kenya", "kenya"),
    _mfg_C("Nigeria", "nigeria"),
    _mfg_C("European Union", "european union", "union europeenne", "eu", "u e", "union europea", "europe", "europa"),
]:
    COUNTRY_EN.update(_d)
COUNTRY_EN["россия"] = "Russia"  # häufigste kyrillische Variante

def _mfg_R(country, *keys):
    return {_mfg_key(k): country for k in keys}

# Hochfrequente Regionen/Städte -> Land (nur für die Erkennungs-Quote; Wert bleibt Eigenname)
PLACE_COUNTRY = {}
for _d in [
    _mfg_R("France", "bretagne", "bzh", "normandie", "basse normandie", "haute normandie", "pays de la loire",
       "provence", "provence alpes cote d azur", "alsace", "rhone alpes", "auvergne rhone alpes", "occitanie",
       "bourgogne", "bourgogne franche comte", "aquitaine", "nouvelle aquitaine", "grand est", "hauts de france",
       "ile de france", "nord pas de calais", "centre val de loire", "midi pyrenees", "languedoc roussillon",
       "franche comte", "limousin", "picardie", "champagne ardenne", "lorraine", "poitou charentes", "corse",
       "finistere", "morbihan", "ille et vilaine", "cotes d armor", "loire atlantique", "vendee", "sarthe",
       "maine et loire", "mayenne", "isere", "savoie", "haute savoie", "calvados", "manche", "orne", "nord",
       "loiret", "rhone", "gironde", "herault", "bouches du rhone", "var", "vaucluse", "gard", "drome", "ain",
       "doubs", "jura", "vosges", "moselle", "bas rhin", "haut rhin", "meurthe et moselle", "marne", "aube",
       "yonne", "cote d or", "saone et loire", "puy de dome", "cantal", "allier", "haute loire", "loire",
       "ardeche", "dordogne", "landes", "pyrenees atlantiques", "lot", "aveyron", "tarn", "haute garonne",
       "gers", "aude", "pyrenees orientales", "vienne", "deux sevres", "charente", "charente maritime",
       "indre et loire", "loir et cher", "eure et loir", "eure", "seine maritime", "somme", "pas de calais",
       "aisne", "oise", "ardennes", "meuse", "haute marne", "territoire de belfort", "essonne", "yvelines",
       "val d oise", "seine et marne", "val de marne", "hauts de seine", "seine saint denis",
       "paris", "lyon", "marseille", "bordeaux", "toulouse", "nantes", "lille", "strasbourg", "rennes",
       "quiberon", "durtal", "guadeloupe", "martinique", "la reunion", "reunion", "antilles guyane", "guyane"),
    _mfg_R("Spain", "navarra", "andalucia", "cataluna", "catalunya", "galicia", "murcia", "aragon", "pais vasco",
       "euskadi", "asturias", "cantabria", "la rioja", "castilla y leon", "castilla la mancha", "extremadura",
       "comunidad valenciana", "comunitat valenciana", "canarias", "islas baleares", "islas canarias",
       "madrid", "barcelona", "valencia", "sevilla", "zaragoza", "malaga", "bilbao", "cordoba", "granada",
       "alicante", "gipuzkoa", "bizkaia", "araba", "navarre"),
    _mfg_R("Italy", "lombardia", "piemonte", "toscana", "emilia romagna", "veneto", "sicilia", "campania", "puglia",
       "calabria", "lazio", "liguria", "trentino", "trentino alto adige", "friuli venezia giulia", "marche",
       "abruzzo", "umbria", "basilicata", "molise", "sardegna", "valle d aosta",
       "milano", "roma", "torino", "napoli", "bologna", "firenze", "genova", "parma", "modena", "verona"),
    _mfg_R("Germany", "bayern", "bavaria", "baden wurttemberg", "baden wuerttemberg", "nordrhein westfalen",
       "niedersachsen", "hessen", "sachsen", "rheinland pfalz", "thuringen", "brandenburg", "sachsen anhalt",
       "schleswig holstein", "mecklenburg vorpommern", "saarland", "berlin", "hamburg", "bremen",
       "munchen", "munich", "koln", "frankfurt", "stuttgart", "dusseldorf", "hannover", "nurnberg"),
    _mfg_R("Bolivia", "la paz", "santa cruz", "cochabamba", "el alto", "oruro", "potosi", "sucre", "tarija",
       "santa cruz de la sierra"),
    _mfg_R("Argentina", "buenos aires", "mendoza", "rosario"),
    _mfg_R("Mexico", "ciudad de mexico", "guadalajara", "monterrey", "jalisco"),
    _mfg_R("Switzerland", "geneve", "zurich", "vaud", "valais", "berne", "bern", "ticino"),
    _mfg_R("Belgium", "flandre", "wallonie", "bruxelles", "anvers", "antwerpen"),
    _mfg_R("United Kingdom", "scotland", "wales", "london", "ecosse", "pays de galles"),
    _mfg_R("Portugal", "lisboa", "porto", "madeira", "azores", "acores"),
]:
    PLACE_COUNTRY.update(_d)

_MFG_PREFIX  = re.compile(r"^[a-z]{2}:")
_MFG_SUBKEYS = sorted([k for k in COUNTRY_EN if len(k) >= 4], key=len, reverse=True)
_MFG_SUB_RE  = re.compile(r"\b(" + "|".join(re.escape(k) for k in _MFG_SUBKEYS) + r")\b")

def _mfg_clean_value(val):
    """-> (bereinigter String oder None, erkannt-bool) für EINEN nicht-leeren Wert."""
    parts, seen, recognized = [], set(), False
    for raw in str(val).split(","):
        tok = raw.strip()
        if not tok or tok.lower() in INVALID:
            continue
        k = _mfg_key(tok)
        if not k:
            continue
        if k in COUNTRY_EN:                       # bekanntes Land -> englischer Name
            disp = COUNTRY_EN[k]; recognized = True
        else:                                     # Region/Stadt/Firma -> als Title-Case-Eigenname behalten
            disp = re.sub(r"\s+", " ", re.sub(r"[-_]", " ", _MFG_PREFIX.sub("", tok.lower()))).strip().title()
            if k in PLACE_COUNTRY or _MFG_SUB_RE.search(k):
                recognized = True
        if disp and disp.lower() not in seen:     # je Eintrag case-insensitiv deduplizieren
            seen.add(disp.lower()); parts.append(disp)
    return (",".join(parts), recognized) if parts else (None, False)

def clean_manufacturing(series):
    """Übersetzt Ländernamen -> Englisch, behält Regionen/Städte als Eigennamen.
    Schnell via Unique-Value-Mapping. -> (bereinigte Serie, erkannt-bool-Serie)."""
    s = series.astype("string")
    cmap = {v: _mfg_clean_value(v) for v in pd.unique(s.dropna())}
    cleaned    = s.map(lambda v: cmap[v][0] if v in cmap else pd.NA).astype("string")
    recognized = s.map(lambda v: cmap[v][1] if v in cmap else False).fillna(False).astype(bool)
    return cleaned, recognized



# ════════════════════════════════════════════════════════════════════════════════
# TEIL A — Reproduktion der bereits gereinigten Spalten (Logik aus Meilestein2_DatenBereinigung)
# ════════════════════════════════════════════════════════════════════════════════
section_header("Teil A  ·  product_name · packaging · additives · ingredients · manufacturing")

# Product Name — erst aus Kurz-/Oberbegriff auffuellen (Deniz), dann Text saeubern (uber6prozent)
product_name_before = df["product_name"].notna().sum()
df["product_name"] = df["product_name"].fillna(df["abbreviated_product_name"]).fillna(df["generic_name"])
df["product_name"] = (
    clean_series(df["product_name"])
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)
merge_report("product_name", product_name_before, df["product_name"].notna().sum())
df = df.drop(columns=["abbreviated_product_name", "generic_name"])

# Packaging — REINHEIT: nur strukturierte Quellen, Praefixe -> lesbar (entfernt). Wie Hauptnotebook.
packaging_before = df["packaging_en"].notna().sum()
df["packaging_en"] = clean_series(df["packaging_en"].astype(str).where(df["packaging_en"].notna(), np.nan))
df["packaging_en"] = df["packaging_en"].fillna(normalize_tags(df["packaging_tags"]))
merge_report("packaging_en", packaging_before, df["packaging_en"].notna().sum())
df = df.drop(columns=["packaging", "packaging_tags", "packaging_text"])

# Additives — lesbare englische Spalte als Survivor, aus Tags auffuellen, Tags verwerfen
additives_before = df["additives_en"].notna().sum()
df["additives_en"] = clean_series(df["additives_en"].astype(str).where(df["additives_en"].notna(), np.nan))
df["additives_en"] = df["additives_en"].fillna(df["additives_tags"])
merge_report("additives_en", additives_before, df["additives_en"].notna().sum())
df = df.drop(columns=["additives"])  # additives_tags BEHALTEN — wird im NOVA-Teil gebraucht

# Ingredients — Tags sauber & zaehlbar halten; ingredients_text als eigene Spalte ERHALTEN
df["ingredients_tags"] = clean_series(df["ingredients_tags"].astype(str).where(df["ingredients_tags"].notna(), np.nan))
df["ingredients_text"] = df["ingredients_text"].astype("string").str.strip()
print(f"  ingredients_tags sauber gehalten ({df['ingredients_tags'].notna().sum():,})  ·  ingredients_text erhalten ({df['ingredients_text'].notna().sum():,})")

# Manufacturing Places — Rohtext auffuellen, dann Ländernamen -> Englisch übersetzen (Regionen/Städte bleiben)
manufacturing_before = df["manufacturing_places_tags"].notna().sum()
df["manufacturing_places_tags"] = df["manufacturing_places_tags"].fillna(df["manufacturing_places"])
df = df.drop(columns=["manufacturing_places"])
df["manufacturing_places_tags"], _mfg_recognized = clean_manufacturing(df["manufacturing_places_tags"])
_mfg_nn  = int(df["manufacturing_places_tags"].notna().sum())
_mfg_rec = int(_mfg_recognized.sum())
merge_report("manufacturing_places", manufacturing_before, _mfg_nn)
print(f"  {'-> davon einem Land zugeordnet':<26}  [Englisch]   {_mfg_rec / _mfg_nn * 100:5.1f}%   erkannt {_mfg_rec:,} / {_mfg_nn:,}   (Ziel >=80%)")


# ════════════════════════════════════════════════════════════════════════════════
# TEIL B — Reproduktion der >6%-Spalten (Logik aus uber6prozent), Praefixe BEHALTEN
# ════════════════════════════════════════════════════════════════════════════════
section_header("Teil B  ·  ingredients_analysis_tags · nova_group · nutrient_levels_tags")

# Ingredients Analysis Tags — kommaseparierte en:-Tags; Junk weg, kleinschreiben, Kommas normalisieren
iat_before = df["ingredients_analysis_tags"].notna().sum()
df["ingredients_analysis_tags"] = (
    clean_series(df["ingredients_analysis_tags"]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("ingredients_analysis_tags", iat_before, df["ingredients_analysis_tags"].notna().sum())

# NOVA Group — numerisch erzwingen, nur gueltige Gruppen 1-4 behalten (nullable Int)
nova_before = df["nova_group"].notna().sum()
_nova = pd.to_numeric(df["nova_group"], errors="coerce")
df["nova_group"] = _nova.where(_nova.isin([1, 2, 3, 4])).astype("Int64")
clean_report("nova_group", nova_before, df["nova_group"].notna().sum())

# Nutrient Levels Tags — wie ingredients_analysis_tags
nlt_before = df["nutrient_levels_tags"].notna().sum()
df["nutrient_levels_tags"] = (
    clean_series(df["nutrient_levels_tags"]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("nutrient_levels_tags", nlt_before, df["nutrient_levels_tags"].notna().sum())


# ════════════════════════════════════════════════════════════════════════════════
# TEIL C — NEU: cities_tags · owner · traces · traces_tags · traces_en  (Praefixe BEHALTEN)
# ════════════════════════════════════════════════════════════════════════════════
section_header("Teil C (neu)  ·  cities_tags · owner · traces · traces_tags · traces_en")

# Tag-Spalten: Junk weg, kleinschreiben, Komma-Abstaende vereinheitlichen — en:/fr: bleiben erhalten
for col in ["cities_tags", "traces", "traces_tags", "traces_en"]:
    before = df[col].notna().sum()
    df[col] = clean_series(df[col]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
    clean_report(col, before, df[col].notna().sum())

# owner — ID-String: nur saeubern + Whitespace kollabieren; NICHT kleinschreiben, nicht kommasplitten
owner_before = df["owner"].notna().sum()
df["owner"] = clean_series(df["owner"]).str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA)
clean_report("owner", owner_before, df["owner"].notna().sum())


# ── Spalten umbenennen (nur die gemergten Survivors; cities_tags/traces*/owner behalten ihre Namen)
df = df.rename(columns={
    "packaging_en":              "packaging",
    "additives_en":              "additives",
    "ingredients_tags":          "ingredients",
    "manufacturing_places_tags": "manufacturing_places",
})

print(f"\n{'─' * _W}")
print(f"  Zwischenstand: {df.shape[1]} Spalten gesamt  |  {len(df):,} Zeilen")
print(f"{'─' * _W}")


════════════════════════════════════════════════════════════════════
  Teil A  ·  product_name · packaging · additives · ingredients · manufacturing
────────────────────────────────────────────────────────────────────
  product_name                [██████████████████░░]   92.6%   befuellt 4,168,958  (+2,746)
  packaging_en                [█░░░░░░░░░░░░░░░░░░░]    8.4%   befuellt 378,935  (+1)
  additives_en                [███░░░░░░░░░░░░░░░░░]   15.5%   befuellt 697,089  (+0)
  ingredients_tags sauber gehalten (1,277,673)  ·  ingredients_text erhalten (1,280,988)
  manufacturing_places        [░░░░░░░░░░░░░░░░░░░░]    4.9%   befuellt 219,354  (+5)
  -> davon einem Land zugeordnet  [Englisch]    86.8%   erkannt 190,464 / 219,354   (Ziel >=80%)

════════════════════════════════════════════════════════════════════
  Teil B  ·  ingredients_analysis_tags · nova_group · nutrient_levels_tags
────────────────────────────────────────────────────────────────────
  ingredients_analysis_tags   [

## Stichprobe / Sichtprüfung

Übersicht (dtype, Befüllung %, ein Beispiel) je gereinigter Spalte + die ersten 10 echten Werte —
zur schnellen Kontrolle, ob Präfixe erhalten und Werte sauber sind.

In [3]:
# Sample-Uebersicht der in diesem Notebook gereinigten Spalten
pd.set_option("display.max_colwidth", 80)
cols = ["product_name", "packaging", "additives", "ingredients", "ingredients_text",
        "manufacturing_places", "ingredients_analysis_tags", "nova_group", "nutrient_levels_tags",
        "cities_tags", "owner", "traces", "traces_tags", "traces_en"]
cols = [c for c in cols if c in df.columns]

uebersicht = pd.DataFrame({
    "dtype":    df[cols].dtypes.astype(str),
    "fuellung": (df[cols].notna().mean() * 100).round(1).astype(str) + " %",
    "beispiel": {c: (df[c].dropna().iloc[0] if df[c].notna().any() else None) for c in cols},
})
print(uebersicht, "\n")

# Pro Spalte die ersten 10 echten Werte (Praefixe sollten erhalten sein, z. B. en:milk)
n = 10
pd.DataFrame({c: df[c].dropna().head(n).reset_index(drop=True) for c in cols})

                            dtype fuellung  \
product_name               string   92.6 %   
packaging                  string    8.4 %   
additives                  string   15.5 %   
ingredients                string   28.4 %   
ingredients_text           string   28.5 %   
manufacturing_places       string    4.9 %   
ingredients_analysis_tags  string   30.1 %   
nova_group                  Int64   25.1 %   
nutrient_levels_tags       string   33.8 %   
cities_tags                string    2.3 %   
owner                      string    3.1 %   
traces                     string    3.7 %   
traces_tags                string    5.1 %   
traces_en                  string    5.1 %   

                                                                                                  beispiel  
product_name                                                                 limonade artisanale a la rose  
packaging                                        Plastic,Cardboard,fr:boite-en-carton,fr:fi

,product_name,packaging,additives,ingredients,ingredients_text,manufacturing_places,ingredients_analysis_tags,nova_group,nutrient_levels_tags,cities_tags,owner,traces,traces_tags,traces_en
0,limonade artisanale a la rose,"Plastic,Cardboard,fr:boite-en-carton,fr:film-en-plastique","E330 - Citric acid,E955 - Sucralose","en:weizenmehl,en:rapsol,en:speisesalz,en:meersalz,en:fefe,en:gerstenmaizextr...","Weizenmehl, Rapsöl, Speisesalz, 1,7% Meersalz, Fefe, Gerstenmaizextrakt, Säu...",France,"en:palm-oil-content-unknown,en:vegan-status-unknown,en:vegetarian-status-unk...",4,"en:fat-in-low-quantity,en:saturated-fat-in-low-quantity,en:sugars-in-low-qua...","lamotte-beuvron-loir-et-cher-france,lancome-loir-et-cher-france",org-le-picoreur-bodin-bio,"en:nuts,en:soybeans","en:nuts,en:soybeans","nuts,soybeans"
1,m&amp;m white,Pot-plastique,E955 - Sucralose,"en:thiamin,en:biotin,en:vitamins,en:chromium,en:minerals,en:garcinia-cambogi...","Thiamin, Biotin, Chromium, Garcinia cambogia fruit extract, Taurine, Green c...","Saint Yrieix,France","en:may-contain-palm-oil,en:vegan-status-unknown,en:vegetarian-status-unknown",4,"en:fat-in-low-quantity,en:saturated-fat-in-low-quantity,en:sugars-in-low-qua...",magescq-landes-france,org-le-picoreur-bodin-bio,en:milk,en:milk,milk
2,chocolate n3,"Plastic,Cardboard","E331 - Sodium citrates,E422 - Glycerol,E503 - Ammonium carbonates","es:honig-stillende-frauen-nicht-geeignet,es:d-bestrahlung-vermeiden-und-be-t...",HONIG stillende Frauen nicht geeignet. D bestrahlung vermeiden und be trocke...,Canada,"en:palm-oil-content-unknown,en:vegan-status-unknown,en:vegetarian-status-unk...",4,"en:fat-in-low-quantity,en:saturated-fat-in-low-quantity,en:sugars-in-low-qua...",lessay-manche-france,org-le-picoreur-bodin-bio,"en:crustaceans,en:fish,en:gluten,en:molluscs,en:mustard,en:nuts,en:peanuts,e...","en:crustaceans,en:fish,en:gluten,en:molluscs,en:mustard,en:nuts,en:peanuts,e...","crustaceans,fish,gluten,molluscs,mustard,nuts,peanuts,sesame seeds,fr:peut-c..."
3,pâte de fruits,fr:1-boite-en-carton-a-recycler-50-sachets-individuels-a-recycler,"E322 - Lecithins,E322i - Lecithin,E331 - Sodium citrates,E422 - Glycerol,E50...","en:soy-protein-isolate,en:protein,en:plant-protein,en:soy-protein,en:wheat-p...","Sojaproteinisolat, Weizen - protein, Kaffee-Extrakt (6 %), Reisprotein, Erbs...",Argentina,"en:palm-oil-content-unknown,en:vegan-status-unknown,en:vegetarian-status-unk...",4,en:fat-in-low-quantity,guerville-yvelines-france,org-label-non-gmo-project,en:nuts,en:nuts,nuts
4,paleta gran reserva - sierra nevada-,"fr:boite-en-carton,fr:film-en-plastique","E415 - Xanthan gum,E955 - Sucralose","en:wheat-flour,en:cereal,en:flour,en:wheat,en:cereal-flour,en:sugar,en:added...","Farine de blé 33%, sucre, huile de colza, œufs de poules élevées en plein ai...",United Kingdom,"en:vegan,en:vegetarian",1,"en:fat-in-moderate-quantity,en:saturated-fat-in-high-quantity,en:sugars-in-l...",saint-agreve-ardeche-france,org-label-non-gmo-project,en:nuts,en:nuts,nuts
5,confiture extra citron de menton,de:packung-en,"E322 - Lecithins,E322i - Lecithin,E331 - Sodium citrates,E422 - Glycerol,E50...","en:water,en:leptospermum-scoparium-mel,en:sodium-c14-c16-olefin-sulfonate,en...","Water, Leptospermum Scoparium Mel (Manuka Honey), Sodium C14-C16 Olefin Sulf...","87500,France","en:palm-oil-free,en:non-vegan,en:vegetarian-status-unknown",4,"en:fat-in-high-quantity,en:saturated-fat-in-high-quantity,en:sugars-in-high-...",pontet-vaucluse-france,org-label-non-gmo-project,en:na,en:nuts,nuts
6,6666,Plastic,"E140 - Chlorophylls and Chlorophyllins,E140i - Chlorophylls,E330 - Citric ac...","en:wheat-flour,en:cereal,en:flour,en:wheat,en:cereal-flour,en:milk-chocolate...","Farine de blé 27%, chocolat au lait 18% (sucre, beurre de cacao, lait entier...",United States,"en:may-contain-palm-oil,en:non-vegan,en:vegetarian-status-unknown",4,"en:fat-in-moderate-quantity,en:salt-in-moderate-quantity",saint-andiol-bouches-du-rhone-france,org-label-non-gmo-project,en:nu

## Detail-Profil: `traces_tags` & `traces_en`

`spalte_info()` zeigt je Spalte dtype, Befüllung, einzigartige Werte und — auf **Tag-Ebene**
(an Kommas gesplittet) — Tags/Eintrag, Top-Werte, Top-Einzel-Tags, Beispiele.

In [4]:
# Detail-Profil der zwei Spalten traces_tags und traces_en — jeweils EINZELN
pd.set_option("display.max_colwidth", 100)

def spalte_info(col, top=15):
    s = df[col]
    n, nn = len(s), df[col].notna().sum()
    print("═" * 70)
    print(f"  Spalte: {col}")
    print("─" * 70)
    print(f"  dtype                         : {s.dtype}")
    print(f"  Zeilen gesamt                 : {n:,}")
    print(f"  befuellt                      : {nn:,}  ({nn/n*100:.2f} %)")
    print(f"  fehlend                       : {n-nn:,}  ({(n-nn)/n*100:.2f} %)")
    print(f"  einzigartige Werte (Strings)  : {s.nunique(dropna=True):,}")

    # Tag-Ebene: an Kommas splitten
    teile = s.dropna().str.split(",")
    tags  = teile.explode().str.strip()
    tags  = tags[tags != ""]
    n_pro = teile.apply(len)
    print(f"  einzelne Tags (mit Wdh.)      : {len(tags):,}")
    print(f"  einzigartige einzelne Tags    : {tags.nunique():,}")
    print(f"  Tags pro Eintrag              : min {n_pro.min()}, median {int(n_pro.median())}, "
          f"max {n_pro.max()}, oe {n_pro.mean():.2f}")

    print(f"\n  Top {top} ganze Werte:")
    print(s.value_counts(dropna=True).head(top).to_string())
    print(f"\n  Top {top} einzelne Tags:")
    print(tags.value_counts().head(top).to_string())
    print(f"\n  Beispielwerte (erste {top}):")
    for v in s.dropna().head(top):
        print(f"    {v}")
    print()

spalte_info("traces_tags")
spalte_info("traces_en")

══════════════════════════════════════════════════════════════════════
  Spalte: traces_tags
──────────────────────────────────────────────────────────────────────
  dtype                         : string
  Zeilen gesamt                 : 4,501,357
  befuellt                      : 228,342  (5.07 %)
  fehlend                       : 4,273,015  (94.93 %)
  einzigartige Werte (Strings)  : 16,949
  einzelne Tags (mit Wdh.)      : 639,001
  einzigartige einzelne Tags    : 11,138
  Tags pro Eintrag              : min 1, median 2, max 27, oe 2.80

  Top 15 ganze Werte:
traces_tags
en:nuts                        21864
en:milk                        10298
en:soybeans                     8250
en:nuts,en:peanuts              7176
en:gluten                       6956
en:nuts,en:soybeans             5164
en:milk,en:nuts                 5070
en:gluten,en:nuts               4516
en:eggs                         4076
en:sesame-seeds                 3282
en:celery                       3268
en:nuts,en:

## Traces-Kreuztabelle + Identität

Vergleicht die drei traces-Spalten: Befüllungs-Muster, 2×2-Kreuztabelle, und wie oft sie **identisch**
sind (wörtlich vs. semantisch = Präfixe weg + sortiert). Kernbefund: `traces_tags` und `traces_en`
sind zu 100 % dieselbe Information in zwei Formaten.

In [5]:
# ════════════════════════════════════════════════════════════════════════════════
# Traces-Kreuztabelle (traces / traces_tags / traces_en) + Identitaet ueber alle 3
# Nach der clean-Zelle ausfuehren — arbeitet auf den gereinigten Spalten.
# ════════════════════════════════════════════════════════════════════════════════
trace_cols = ["traces", "traces_tags", "traces_en"]

# ── 1) Verfuegbarkeits-Kreuztabelle: welche Kombination der 3 Spalten ist befuellt? ──
praesenz = df[trace_cols].notna()
muster = (praesenz.value_counts()
          .reset_index(name="anzahl_produkte")
          .sort_values("anzahl_produkte", ascending=False))
muster["anteil_%"] = (muster["anzahl_produkte"] / len(df) * 100).round(3)
print("Befuellungs-Muster ueber traces / traces_tags / traces_en  (True = befuellt):")
print(muster.to_string(index=False))

print("\n2x2-Kreuztabelle 'befuellt' (traces_tags x traces_en):")
print(pd.crosstab(df["traces_tags"].notna(), df["traces_en"].notna(),
                  rownames=["traces_tags"], colnames=["traces_en"], margins=True))

# ── 2) Identische Inhalte ueber alle 3 Spalten ──────────────────────────────────────
alle = praesenz.all(axis=1)
print(f"\nProdukte mit allen 3 Spalten befuellt: {alle.sum():,}")

# a) WOERTLICH identisch (exakt gleiche Strings)
literal = (alle & df["traces"].eq(df["traces_tags"]) & df["traces_tags"].eq(df["traces_en"])).fillna(False)
print(f"  a) woertlich identisch in allen 3 (exakt):           {literal.sum():,}")

# b) SEMANTISCH identisch: Praefixe (en:/fr:/es: ...) weg, Bindestrich->Space, Tags sortiert
def normiere(s):
    s = (s.astype("string").str.lower()
           .str.replace(r"\b[a-z]{2}:", "", regex=True)   # Sprach-/Taxonomie-Praefixe weg
           .str.replace(r"[-_]", " ", regex=True)
           .str.replace(r"\s+", " ", regex=True).str.strip())
    return s.str.split(",").apply(
        lambda x: ",".join(sorted(t.strip() for t in x if t.strip())) if isinstance(x, list) else pd.NA)

n_tr, n_tg, n_en = normiere(df["traces"]), normiere(df["traces_tags"]), normiere(df["traces_en"])
semantisch = (alle & n_tr.eq(n_tg) & n_tg.eq(n_en)).fillna(False)
print(f"  b) semantisch identisch (Praefix/Reihenfolge egal):  {semantisch.sum():,}")

# ── 3) Paarweise semantische Uebereinstimmung (jeweils wo BEIDE befuellt) ────────────
print("\nPaarweise semantische Uebereinstimmung (wo beide befuellt):")
norm = {"traces": n_tr, "traces_tags": n_tg, "traces_en": n_en}
for a, b in [("traces", "traces_tags"), ("traces", "traces_en"), ("traces_tags", "traces_en")]:
    beide = (df[a].notna() & df[b].notna())
    gleich = (beide & norm[a].eq(norm[b])).fillna(False)
    quote = f"{gleich.sum()/beide.sum()*100:.1f} %" if beide.sum() else "—"
    print(f"  {a:12s} == {b:12s}:  {gleich.sum():,} von {beide.sum():,}  ({quote})")

Befuellungs-Muster ueber traces / traces_tags / traces_en  (True = befuellt):
 traces  traces_tags  traces_en  anzahl_produkte  anteil_%
  False        False      False          4273015    94.927
   True         True       True           166328     3.695
  False         True       True            62008     1.378
   True         True      False                6     0.000

2x2-Kreuztabelle 'befuellt' (traces_tags x traces_en):
traces_en      False    True      All
traces_tags                          
False        4273015       0  4273015
True               6  228336   228342
All          4273021  228336  4501357

Produkte mit allen 3 Spalten befuellt: 166,328
  a) woertlich identisch in allen 3 (exakt):           1,304
  b) semantisch identisch (Praefix/Reihenfolge egal):  155,791

Paarweise semantische Uebereinstimmung (wo beide befuellt):
  traces       == traces_tags :  155,795 von 166,334  (93.7 %)
  traces       == traces_en   :  155,791 von 166,328  (93.7 %)
  traces_tags  == trac

## Diagnose: wie viele Produkte sind klassifizierbar?

Zählt Produkte mit `ingredients` **und** `categories_tags` — die theoretische Obergrenze für eine
regelbasierte NOVA-Ableitung (~23,9 %).

In [6]:
can_attempt = (
    df["ingredients"].notna() & 
    df["categories_tags"].notna()
).sum()
print(f"Kandidaten für Regelableitung: {can_attempt / len(df):.1%}")

Kandidaten für Regelableitung: 23.9%


## Diagnose: neue Kandidaten

Produkte **ohne** `nova_group`, aber **mit** Zutaten + Kategorie → die realistisch befüllbare Menge
(~117.709 / 2,6 %).

In [7]:
new_candidates = (
    df["nova_group"].isna() &
    df["ingredients"].notna() &
    df["categories_tags"].notna()
).sum()

print(f"Neue Kandidaten (nova_group NaN + Inputs vorhanden): {new_candidates:,}")
print(f"Anteil am Gesamtdatensatz: {new_candidates / len(df):.1%}")

Neue Kandidaten (nova_group NaN + Inputs vorhanden): 117,709
Anteil am Gesamtdatensatz: 2.6%


# NOVA-Ableitung — arbeitet auf DEMSELBEN `df`

Ab hier wird für Produkte ohne NOVA eine Gruppe 1–4 abgeleitet (Port der echten OFF-Logik).
**Kein Neuladen** mehr — die in der Bereinigung erzeugten Spalten bleiben erhalten.

**Setup:** Re-Import + `Counter` (für Häufigkeiten).

In [8]:
import pandas as pd
import numpy as np
from collections import Counter
pd.set_option("display.max_columns", None)

## NOVA-Vorbereitung (kein Neuladen)

Normalisiert nur die Matching-Spalten in Tag-Form: `categories_tags` (roh) und `additives_tags`
(roh behalten) → lowercase + Komma-Abstände, Präfixe bleiben. Die Zutaten kommen aus der bereits
gereinigten Spalte `ingredients`. `nova_group` ist schon aus Teil B sauber (1–4).

In [9]:
# NOVA-Vorbereitung — KEIN Neuladen: wir arbeiten auf DEMSELBEN df wie die Bereinigung.
# Die fuer das Marker-Matching noetigen TAG-Spalten sind hier:
#   categories -> categories_tags (roh geladen) · ingredients -> "ingredients" (gereinigte Tags)
#   additives  -> additives_tags  (roh behalten)
# categories_tags + additives_tags noch normalisieren (lowercase, Komma-Abstaende; Praefixe bleiben).
# nova_group wurde bereits in Teil B auf 1-4/Int64 gesaeubert.
for c in ["categories_tags", "additives_tags"]:
    df[c] = clean_series(df[c]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)

print(f"Zeilen gesamt        : {len(df):,}")
print(f"nova_group offiziell : {df['nova_group'].notna().sum():,}  ({df['nova_group'].notna().mean()*100:.1f} %)")
for tt, c in [("categories", "categories_tags"), ("ingredients", "ingredients"), ("additives", "additives_tags")]:
    print(f"{c:16s} : {df[c].notna().sum():,}  ({df[c].notna().mean()*100:.1f} %)")

Zeilen gesamt        : 4,501,357
nova_group offiziell : 1,129,053  (25.1 %)
categories_tags  : 1,848,193  (41.1 %)
ingredients      : 1,277,673  (28.4 %)
additives_tags   : 697,089  (15.5 %)


## Marker-Wörterbuch (`NOVA_MARKER`, 327 Einträge)

1:1 aus Open Food Facts portiert (`Config_off.pm` + Taxonomien). Schlüssel `"tagtype/tagid"` → Gruppe.
**Additive direkt per E-Nummer** (`additives/en:e621 → 4`), kein Klassen-Umweg. 30 Kategorien,
57 Zutaten, 240 Additive.

In [10]:
# NOVA-Marker, 1:1 portiert aus Open Food Facts (Stand 2026-06-21):
#   * lib/ProductOpener/Config_off.pm  ->  $options{nova_groups_tags}   (Kern, inkl. Additive je E-Nummer)
#   * taxonomies/food/{categories,ingredients}.txt  ->  nova:en: Eigenschaft (zusaetzliche Marker)
# Schluessel = "tagtype/tagid", Wert = NOVA-Gruppe (2/3/4). Bei Konflikt wurde die hoehere Gruppe behalten.
# Hinweis: Additiv-KLASSEN-Marker in der Taxonomie sind dort auskommentiert/inaktiv -> Additive
# werden (wie im echten OFF) direkt ueber die E-Nummer gematcht, KEIN Klassen-Umweg.
NOVA_MARKER = {
    # --- categories (30) ---
    "categories/en:alcoholic-beverages": 3, "categories/en:animal-fats": 2, "categories/en:baby-milks": 3,
    "categories/en:beers": 3, "categories/en:candies": 3, "categories/en:cheeses": 3,
    "categories/en:chocolates": 3, "categories/en:ciders": 3, "categories/en:fats": 2,
    "categories/en:hard-liquors": 3, "categories/en:honeys": 2, "categories/en:ice-creams": 3,
    "categories/en:maple-syrups": 2, "categories/en:meal-kits": 3, "categories/en:meals": 3,
    "categories/en:pates": 3, "categories/en:prepared-meats": 3, "categories/en:salts": 2,
    "categories/en:salty-snacks": 3, "categories/en:sandwiches": 3, "categories/en:sausages": 3,
    "categories/en:sodas": 3, "categories/en:starches": 2, "categories/en:sugars": 2,
    "categories/en:sugary-snacks": 3, "categories/en:sweet-snacks": 3, "categories/en:terrines": 3,
    "categories/en:tofu": 3, "categories/en:vinegars": 2, "categories/en:wines": 3,
    # --- ingredients (57) ---
    "ingredients/en:anti-caking-agent": 3, "ingredients/en:anti-foaming-agent": 4, "ingredients/en:bread": 3,
    "ingredients/en:bulking-agent": 4, "ingredients/en:butter": 3, "ingredients/en:carbonating-agent": 4,
    "ingredients/en:casein": 4, "ingredients/en:cheese": 3, "ingredients/en:colour": 4,
    "ingredients/en:colour-stabilizer": 4, "ingredients/en:concentrated-whey-protein": 4,
    "ingredients/en:dextrose": 4, "ingredients/en:emulsifier": 4, "ingredients/en:firming-agent": 4,
    "ingredients/en:flavour": 4, "ingredients/en:flavour-enhancer": 4, "ingredients/en:flavouring": 4,
    "ingredients/en:fructose": 4, "ingredients/en:fruit-juice-concentrate": 4,
    "ingredients/en:gelling-agent": 4, "ingredients/en:glazing-agent": 4, "ingredients/en:glucose": 4,
    "ingredients/en:glucose-syrup": 4, "ingredients/en:gluten": 4,
    "ingredients/en:high-fructose-corn-syrup": 4, "ingredients/en:honey": 3, "ingredients/en:humectant": 4,
    "ingredients/en:hydrogenated-fat": 4, "ingredients/en:hydrogenated-oil": 4,
    "ingredients/en:hydrolysed-cereal": 4, "ingredients/en:hydrolysed-proteins": 4,
    "ingredients/en:invert-sugar": 4, "ingredients/en:lactose": 4, "ingredients/en:lecithin": 4,
    "ingredients/en:maltodextrin": 4, "ingredients/en:maple-syrup": 3,
    "ingredients/en:mechanically-separated-meat": 4, "ingredients/en:milk-powder": 3,
    "ingredients/en:milk-proteins": 4, "ingredients/en:modified-flour": 4,
    "ingredients/en:modified-starch": 4, "ingredients/en:preservative": 3, "ingredients/en:salt": 3,
    "ingredients/en:sauce": 3, "ingredients/en:sequestrant": 4, "ingredients/en:soy-preparation": 4,
    "ingredients/en:starch": 3, "ingredients/en:sugar": 3, "ingredients/en:sweetener": 4,
    "ingredients/en:thickener": 4, "ingredients/en:vegetable-fat": 3, "ingredients/en:vegetable-fiber": 4,
    "ingredients/en:vegetable-oil": 3, "ingredients/en:vegetal-oil": 3, "ingredients/en:whey": 4,
    "ingredients/en:whey-product": 4, "ingredients/en:whey-proteins": 4,
    # --- additives (240) ---
    "additives/en:e100": 4, "additives/en:e101": 4, "additives/en:e101a": 4, "additives/en:e102": 4,
    "additives/en:e103": 4, "additives/en:e104": 4, "additives/en:e105": 4, "additives/en:e106": 4,
    "additives/en:e107": 4, "additives/en:e110": 4, "additives/en:e1104": 4, "additives/en:e111": 4,
    "additives/en:e120": 4, "additives/en:e121": 4, "additives/en:e122": 4, "additives/en:e123": 4,
    "additives/en:e124": 4, "additives/en:e125": 4, "additives/en:e126": 4, "additives/en:e127": 4,
    "additives/en:e128": 4, "additives/en:e129": 4, "additives/en:e130": 4, "additives/en:e131": 4,
    "additives/en:e132": 4, "additives/en:e133": 4, "additives/en:e140": 4, "additives/en:e1400": 4,
    "additives/en:e1401": 4, "additives/en:e1402": 4, "additives/en:e1403": 4, "additives/en:e1404": 4,
    "additives/en:e1405": 4, "additives/en:e141": 4, "additives/en:e1410": 4, "additives/en:e1412": 4,
    "additives/en:e1413": 4, "additives/en:e1414": 4, "additives/en:e142": 4, "additives/en:e1420": 4,
    "additives/en:e1422": 4, "additives/en:e143": 4, "additives/en:e1440": 4, "additives/en:e1442": 4,
    "additives/en:e1450": 4, "additives/en:e1451": 4, "additives/en:e14xx": 4, "additives/en:e150": 4,
    "additives/en:e1505": 4, "additives/en:e150a": 4, "additives/en:e150b": 4, "additives/en:e150c": 4,
    "additives/en:e150d": 4, "additives/en:e151": 4, "additives/en:e152": 4, "additives/en:e1521": 4,
    "additives/en:e153": 4, "additives/en:e154": 4, "additives/en:e155": 4, "additives/en:e15x": 4,
    "additives/en:e160": 4, "additives/en:e160a": 4, "additives/en:e160b": 4, "additives/en:e160c": 4,
    "additives/en:e160d": 4, "additives/en:e160e": 4, "additives/en:e160f": 4, "additives/en:e161": 4,
    "additives/en:e161a": 4, "additives/en:e161b": 4, "additives/en:e161c": 4, "additives/en:e161d": 4,
    "additives/en:e161e": 4, "additives/en:e161f": 4, "additives/en:e161g": 4, "additives/en:e161h": 4,
    "additives/en:e161i": 4, "additives/en:e161j": 4, "additives/en:e162": 4, "additives/en:e163": 4,
    "additives/en:e163a": 4, "additives/en:e163b": 4, "additives/en:e163c": 4, "additives/en:e163d": 4,
    "additives/en:e163e": 4, "additives/en:e163f": 4, "additives/en:e164": 4, "additives/en:e165": 4,
    "additives/en:e166": 4, "additives/en:e170": 4, "additives/en:e171": 4, "additives/en:e172": 4,
    "additives/en:e173": 4, "additives/en:e174": 4, "additives/en:e175": 4, "additives/en:e180": 4,
    "additives/en:e181": 4, "additives/en:e182": 4, "additives/en:e202": 3, "additives/en:e249": 3,
    "additives/en:e250": 3, "additives/en:e251": 3, "additives/en:e252": 3, "additives/en:e290": 4,
    "additives/en:e322": 4, "additives/en:e325": 4, "additives/en:e326": 4, "additives/en:e327": 4,
    "additives/en:e328": 4, "additives/en:e329": 4, "additives/en:e400": 4, "additives/en:e401": 4,
    "additives/en:e402": 4, "additives/en:e403": 4, "additives/en:e404": 4, "additives/en:e405": 4,
    "additives/en:e406": 4, "additives/en:e407": 4, "additives/en:e407a": 4, "additives/en:e409": 4,
    "additives/en:e410": 4, "additives/en:e412": 4, "additives/en:e413": 4, "additives/en:e414": 4,
    "additives/en:e415": 4, "additives/en:e416": 4, "additives/en:e417": 4, "additives/en:e418": 4,
    "additives/en:e420": 4, "additives/en:e421": 4, "additives/en:e422": 4, "additives/en:e425": 4,
    "additives/en:e428": 4, "additives/en:e430": 4, "additives/en:e431": 4, "additives/en:e432": 4,
    "additives/en:e433": 4, "additives/en:e434": 4, "additives/en:e435": 4, "additives/en:e436": 4,
    "additives/en:e440": 4, "additives/en:e441": 4, "additives/en:e442": 4, "additives/en:e443": 4,
    "additives/en:e444": 4, "additives/en:e445": 4, "additives/en:e450": 4, "additives/en:e451": 4,
    "additives/en:e452": 4, "additives/en:e459": 4, "additives/en:e460": 4, "additives/en:e461": 4,
    "additives/en:e463": 4, "additives/en:e464": 4, "additives/en:e465": 4, "additives/en:e466": 4,
    "additives/en:e468": 4, "additives/en:e469": 4, "additives/en:e470": 4, "additives/en:e470a": 4,
    "additives/en:e470b": 4, "additives/en:e471": 4, "additives/en:e472a": 4, "additives/en:e472b": 4,
    "additives/en:e472c": 4, "additives/en:e472d": 4, "additives/en:e472e": 4, "additives/en:e472f": 4,
    "additives/en:e473": 4, "additives/en:e474": 4, "additives/en:e475": 4, "additives/en:e476": 4,
    "additives/en:e477": 4, "additives/en:e478": 4, "additives/en:e479b": 4, "additives/en:e480": 4,
    "additives/en:e481": 4, "additives/en:e482": 4, "additives/en:e483": 4, "additives/en:e491": 4,
    "additives/en:e492": 4, "additives/en:e493": 4, "additives/en:e494": 4, "additives/en:e495": 4,
    "additives/en:e551": 4, "additives/en:e620": 4, "additives/en:e621": 4, "additives/en:e622": 4,
    "additives/en:e623": 4, "additives/en:e624": 4, "additives/en:e625": 4, "additives/en:e626": 4,
    "additives/en:e627": 4, "additives/en:e628": 4, "additives/en:e629": 4, "additives/en:e630": 4,
    "additives/en:e631": 4, "additives/en:e632": 4, "additives/en:e633": 4, "additives/en:e634": 4,
    "additives/en:e635": 4, "additives/en:e636": 4, "additives/en:e637": 4, "additives/en:e640": 4,
    "additives/en:e641": 4, "additives/en:e650": 4, "additives/en:e900": 4, "additives/en:e900a": 4,
    "additives/en:e901": 4, "additives/en:e902": 4, "additives/en:e903": 4, "additives/en:e904": 4,
    "additives/en:e905": 4, "additives/en:e905c": 4, "additives/en:e905d": 4, "additives/en:e907": 4,
    "additives/en:e938": 4, "additives/en:e939": 4, "additives/en:e941": 4, "additives/en:e942": 4,
    "additives/en:e943a": 4, "additives/en:e943b": 4, "additives/en:e950": 4, "additives/en:e951": 4,
    "additives/en:e952": 4, "additives/en:e953": 4, "additives/en:e954": 4, "additives/en:e955": 4,
    "additives/en:e956": 4, "additives/en:e957": 4, "additives/en:e959": 4, "additives/en:e960": 4,
    "additives/en:e961": 4, "additives/en:e962": 4, "additives/en:e964": 4, "additives/en:e965": 4,
    "additives/en:e966": 4, "additives/en:e967": 4, "additives/en:e968": 4, "additives/en:e969": 4,
}

print("Marker gesamt:", len(NOVA_MARKER), dict(Counter(k.split("/")[0] for k in NOVA_MARKER)))
print("Gruppen-Verteilung der Marker:", dict(Counter(NOVA_MARKER.values())))

Marker gesamt: 327 {'categories': 30, 'ingredients': 57, 'additives': 240}
Gruppen-Verteilung der Marker: {3: 42, 2: 8, 4: 277}


## Klassifikator + Gates

- `entscheide(groups)` — Kernregel: **4 schlägt alles**; eine **Gruppe-2**-Markierung bleibt 2
  (nicht hochgestuft); sonst 3; sonst 1.
- `_matched` / `klassifiziere` — Tags splitten, im `NOVA_MARKER` nachschlagen, je Produkt die
  Treffergruppen sammeln und entscheiden.
- `gate` — nur Produkte mit **Zutaten UND Kategorie** (kein Non-Food) werden bewertet.
- `NOVA_COLS` ordnet die Logik-Begriffe den echten Spaltennamen im vereinigten `df` zu.

In [11]:
# ── Klassifikator: treuer Port von compute_nova_group (Food.pm) ──────────────────────────────
# In diesem (vereinigten) df heisst die gereinigte Zutaten-Tag-Spalte "ingredients".
NOVA_COLS = {"categories": "categories_tags", "ingredients": "ingredients", "additives": "additives_tags"}

# Start bei Gruppe 1; hoechste getroffene Markergruppe gewinnt, ABER eine Gruppe-2-Markierung
# wird NICHT von Gruppe 3 hochgestuft ("Zucker bleibt Gruppe 2"). 4 schlaegt immer alles.
def entscheide(groups):
    if 4 in groups: return 4
    if 2 in groups: return 2          # Gruppe-2 vor Gruppe-3 geschuetzt (Food.pm-Regel)
    if 3 in groups: return 3
    return 1                           # Gates erfuellt, aber kein Marker -> minimal verarbeitet

def _matched(frame, tagtype):
    s = frame[NOVA_COLS[tagtype]].dropna().str.split(",").explode().str.strip()
    g = (tagtype + "/" + s).map(NOVA_MARKER)          # dict-map (schnell); NaN wenn kein Marker
    return g.dropna().astype(int)

def klassifiziere(frame):
    # frame muss bereits die Gates erfuellen (Zutaten + Kategorie vorhanden, kein Non-Food)
    mg = pd.concat([_matched(frame, "categories"), _matched(frame, "ingredients"), _matched(frame, "additives")])
    gp = mg.groupby(level=0).agg(lambda x: set(x))     # Menge getroffener Gruppen je Produkt
    res = gp.reindex(frame.index).apply(lambda s: entscheide(s) if isinstance(s, set) else 1)
    return res.astype("Int64")

def gate(frame):
    # Abstinenz-Gates (Food.pm): Zutaten UND Kategorie vorhanden, kein Non-Food.
    # (OFF-Ausnahmen "Wasser/Gruppe-2 ohne Zutaten" und ">=50% unbekannte Zutaten" bewusst nicht repliziert.)
    has_ing = frame[NOVA_COLS["ingredients"]].notna()
    has_cat = frame[NOVA_COLS["categories"]].notna()
    nonfood = frame[NOVA_COLS["categories"]].str.contains("en:non-food-products", regex=False).fillna(False)
    return (has_ing & has_cat & ~nonfood).astype(bool)

print("Klassifikator + Gates definiert.")

Klassifikator + Gates definiert.


## Validierung (Ehrlichkeitsprüfung, Obergrenze)

Wendet die Regeln auf bereits **offiziell** gelabelte Produkte an und vergleicht: Konfusionsmatrix,
Accuracy vs. Baseline, Precision/Recall je Gruppe, Abstinenzrate. **Kill-Bedingung vorab:**
(a) Accuracy ≥ Baseline + 10 pp, sonst nichts ausliefern; (b) nur Gruppen mit Precision ≥ 80 %.
Diese Accuracy ist eine **Obergrenze** (gelabelte Produkte haben sauberere Eingaben).

In [12]:
# ── Validierung gegen die offiziell gelabelten Zeilen (OBERGRENZE der Guete) ─────────────────
# Wichtig: diese Zeilen haben sauberere Eingaben als die Zielprodukte -> Accuracy hier ist ein
# optimistischer Oberwert.
labeled = df[df["nova_group"].notna()].copy()
g_lab   = gate(labeled)
val     = labeled[g_lab].copy()

y_true = val["nova_group"].astype(int)
y_pred = klassifiziere(val).astype(int)

print(f"Gelabelte Zeilen           : {len(labeled):,}")
print(f"davon Gates erfuellt (bewertbar): {int(g_lab.sum()):,}")
print(f"abstiniert (Gate gerissen)  : {int((~g_lab).sum()):,}  ({(~g_lab).mean()*100:.1f} %)\n")

konf = pd.crosstab(y_true, y_pred, rownames=["offiziell"], colnames=["abgeleitet"], dropna=False)
print("Konfusionsmatrix (Zeile = offiziell, Spalte = abgeleitet):")
print(konf, "\n")

acc      = (y_true == y_pred).mean()
mode_cls = int(y_true.mode().iloc[0])
baseline = (y_true == mode_cls).mean()
print(f"Accuracy : {acc*100:.1f} %   |   Baseline (immer {mode_cls}): {baseline*100:.1f} %   |   Delta: {(acc-baseline)*100:+.1f} pp\n")

prec, rec = {}, {}
for k in [1, 2, 3, 4]:
    tp = int(((y_pred == k) & (y_true == k)).sum())
    prec[k] = tp / max(int((y_pred == k).sum()), 1)
    rec[k]  = tp / max(int((y_true == k).sum()), 1)
print("Precision / Recall je Gruppe:")
print(pd.DataFrame({"precision": prec, "recall": rec}).round(3), "\n")

# ── Kill-Bedingung (VORAB festgelegt) ────────────────────────────────────────────────────────
kill_a  = (acc - baseline) >= 0.10                       # (a) Accuracy muss Baseline um >=10pp schlagen
allowed = {k for k in [1, 2, 3, 4] if prec[k] >= 0.80}   # (b) nur Gruppen mit Precision >=80% ausliefern
print(f"Kill (a) Accuracy >= Baseline+10pp : {'OK' if kill_a else 'GERISSEN -> keine Auslieferung'}")
print(f"Kill (b) auslieferbare Gruppen (Precision>=80%): {sorted(allowed) if kill_a else '— (a gerissen)'}")

Gelabelte Zeilen           : 1,129,053
davon Gates erfuellt (bewertbar): 955,164
abstiniert (Gate gerissen)  : 173,889  (15.4 %)

Konfusionsmatrix (Zeile = offiziell, Spalte = abgeleitet):
abgeleitet       1      2       3       4
offiziell                                
1           127330      0       0      13
2                0  19861       0       0
3              700      0  206851      86
4               39      9     464  599811 

Accuracy : 99.9 %   |   Baseline (immer 4): 62.9 %   |   Delta: +37.0 pp

Precision / Recall je Gruppe:
   precision  recall
1      0.994   1.000
2      1.000   1.000
3      0.998   0.996
4      1.000   0.999 

Kill (a) Accuracy >= Baseline+10pp : OK
Kill (b) auslieferbare Gruppen (Precision>=80%): [1, 2, 3, 4]


## Ergebnis schreiben

Legt `nova_group_derived` (nur Kandidaten + nur erlaubte Gruppen) und `nova_group_source`
(`off_precomputed`/`rule_derived`/`<NA>`) an; `nova_group` bleibt unangetastet. Danach
Konsistenz-Checks, Gesamtabdeckung und eine Stichprobe je abgeleiteter Gruppe.

In [13]:
# ── Anwenden auf die Kandidaten (nova fehlt + Gates erfuellt) + zwei neue Spalten ────────────
df["nova_group_derived"] = pd.array([pd.NA] * len(df), dtype="Int64")
df["nova_group_source"]  = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[df["nova_group"].notna(), "nova_group_source"] = "off_precomputed"

cand_mask = df["nova_group"].isna() & gate(df)
print(f"Kandidaten (nova NaN + Gates erfuellt): {int(cand_mask.sum()):,}")

if kill_a and allowed:
    pred = klassifiziere(df[cand_mask])
    pred = pred.where(pred.isin(list(allowed)), pd.NA).astype("Int64")   # nur Precision>=80%-Gruppen
    df.loc[cand_mask, "nova_group_derived"] = pred
    got = df["nova_group_derived"].notna()
    df.loc[got, "nova_group_source"] = "rule_derived"
    print(f"davon abgeleitet & ausgeliefert (Gruppen {sorted(allowed)}): {int(got.sum()):,}")
    print("\nVerteilung abgeleitete Gruppen:")
    print(df.loc[got, "nova_group_derived"].value_counts().sort_index())
else:
    print("Kill-Bedingung (a) gerissen -> KEINE abgeleiteten Werte ausgeliefert (nur Diagnose).")

# Konsistenz-Checks + Gesamtabdeckung
print("\nQuelle-Verteilung (nova_group_source):")
print(df["nova_group_source"].value_counts(dropna=False))
assert (df["nova_group_source"] == "off_precomputed").sum() == int(df["nova_group"].notna().sum())
assert df.loc[df["nova_group_source"] == "rule_derived", "nova_group"].isna().all()
merged = df["nova_group"].fillna(df["nova_group_derived"])
print(f"\nNOVA-Abdeckung: offiziell {df['nova_group'].notna().mean()*100:.1f} %  ->  inkl. abgeleitet {merged.notna().mean()*100:.1f} %")

# Stichprobe je abgeleiteter Gruppe
pd.set_option("display.max_colwidth", 70)
samp = df[df["nova_group_source"] == "rule_derived"]
beispiele = pd.concat([samp[samp["nova_group_derived"] == k].head(3) for k in [1, 2, 3, 4]])
beispiele[["product_name", "categories_tags", "ingredients", "additives_tags", "nova_group_derived","nova_group_source"]]

Kandidaten (nova NaN + Gates erfuellt): 117,299
davon abgeleitet & ausgeliefert (Gruppen [1, 2, 3, 4]): 117,299

Verteilung abgeleitete Gruppen:
nova_group_derived
1    80279
2     5598
3    31396
4       26
Name: count, dtype: Int64

Quelle-Verteilung (nova_group_source):
nova_group_source
<NA>               3255005
off_precomputed    1129053
rule_derived        117299
Name: count, dtype: int64[pyarrow]

NOVA-Abdeckung: offiziell 25.1 %  ->  inkl. abgeleitet 27.7 %


,product_name,categories_tags,ingredients,additives_tags,nova_group_derived,nova_group_source
12,granola bio le chocolaté,"en:plant-based-foods-and-beverages,en:plant-based-foods,en:fruits-...","es:honig-stillende-frauen-nicht-geeignet,es:d-bestrahlung-vermeide...",<NA>,1,rule_derived
31,volle yoghurt,"en:beverages-and-beverages-preparations,en:beverages",en:nik-odżywczy-wspiera-prawidłowe-funkcjonowanie-organizmu-zeskan...,<NA>,1,rule_derived
69,<NA>,"en:beverages-and-beverages-preparations,en:plant-based-foods-and-b...","es:fullstoff,es:zinkglu,es:conat,es:modifizierte-starke,en:biotin,...",<NA>,1,rule_derived
70,<NA>,en:fats,"en:kakaomasse,en:rohrzucker,en:karamellisierte-kokosflocken,en:kak...",<NA>,2,rule_derived
296,huile végétale,"en:plant-based-foods-and-beverages,en:plant-based-foods,en:fats,en...",fr:total-des,<NA>,2,rule_derived
1033,คุกกี้สเปลท์เนยสดผสมข้าวกล้องงอก,"en:dairies,en:snacks,en:fats,en:spreads,en:sweet-snacks,en:biscuit...",th:แป้งสเปลท์ออร์แกนิค-ข้าวกล้องงอก-เนยสด-ไข่ไก่-ไอซ่ง-ผงฟู-กลิ่นว...,<NA>,2,rule_derived
18,powdered peanut butter,"en:snacks,en:meals,en:rice-dishes,en:risottos,en:powder-peanut-butter","en:water,en:leptospermum-scoparium-mel,en:sodium-c14-c16-olefin-su...",<NA>,3,rule_derived
33,anthony's organic cocoa butter chunks,"en:sandwiches,en:wraps",en:allulose,<NA>,3,rule_derived
48,gummie,"en:meals,en:pasta-dishes,en:stuffed-pastas,en:ravioli,en:japanese-...","en:acrylate-adhesive,en:polyester,en:silicone-adhesive",<NA>,3,rule_derived
341652,enduit à cuisson antiadhésif original,en:non-open-products-facts,"en:canola-oil,en:oil-and-fat,en:vegetable-oil-and-fat,en:rapeseed-...","en:e322,en:e322i,en:e943b",4,rule_derived
